# Module 7 Guided Lab: Aggregation, Reshaping, and Light Validation

**BAN 6003: Data Management and Analytics Integration**

In this lab, we return to the `nycflights13` data. This module is practical: we will use Pandas to summarize data, reshape data, and run light validation checks after those transformations.

The key idea for this week is:

> Aggregation and reshaping can change what each row means.

The original `flights` table has one row per flight. After transformation, a row might mean one carrier, one month, one route, or one carrier-month combination.

## Lab Learning Goals

By the end of this lab, you should be able to:

1. Use `groupby()` and `agg()` to summarize flight data.
2. Use `reset_index()` to turn grouped results back into a regular DataFrame.
3. Explain how aggregation changes the unit of analysis.
4. Use `melt()` to reshape wide data into long data.
5. Use `pivot()` or `pivot_table()` to reshape long data into wide data.
6. Run light validation checks after aggregation or reshaping.
7. Write short notes explaining whether the transformed dataset still matches the intended unit of analysis.

We will not use custom functions or complex loops.

## Business Scenario

Imagine you are an analyst helping an airport operations team understand flight delays.

The raw `flights` table has one row per flight. That is useful if we want to inspect individual flights.

But managers usually ask higher-level questions:

- Which carrier has the highest average departure delay?
- Which month has the worst average arrival delay?
- Which origin-destination route has the most flights?
- How do monthly delay patterns compare across carriers?

To answer these questions, we need aggregation and reshaping.

## 0. Setup: Load Packages and Data

The repository requirements include `nycflights13`. In Codespaces they are installed during setup. When working locally, install `requirements.txt` before opening the notebook.

### Reproducibility Note

Package installation is kept outside the analytical workflow. This makes the notebook easier to rerun and avoids hidden environment changes in the middle of an analysis.

In [ ]:
import nycflights13
import pandas as pd
from pathlib import Path

In [ ]:
flights = nycflights13.flights.copy()
flights.head()

### First reminder: What does one row mean?

Before transforming anything, we need to know the current unit of analysis. In the original `flights` table, each row represents one flight.

In [ ]:
flights.shape

In [ ]:
flights.columns

In [ ]:
flights[["year", "month", "day", "carrier", "flight", "origin", "dest", "dep_delay", "arr_delay", "distance", "air_time"]].head()

## 1. Aggregation with `groupby()` and `agg()`

### Business question

Which airline carrier had the highest average departure delay?

To answer this, we need to move from one row per flight to one row per carrier. That is aggregation.

The basic pattern is:

```python
data.groupby("grouping_column").agg({"value_column": "summary_function"})
```

Important arguments:

- The column inside `groupby()` defines the level of the summary.
- The dictionary inside `agg()` says which variable to summarize and how.
- Common summary functions include `"mean"`, `"median"`, `"count"`, `"max"`, `"min"`, and `"std"`.

In [ ]:
carrier_delay = flights.groupby("carrier").agg({
    "dep_delay": "mean"
})

carrier_delay.head()

The result is indexed by `carrier`. This is useful, but it does not look like a regular DataFrame yet. In many workflows, we use `reset_index()` after a grouped summary.

In [ ]:
carrier_delay = flights.groupby("carrier").agg({
    "dep_delay": "mean"
}).reset_index()

carrier_delay.head()

Now we can sort the summary to see which carriers have the highest average departure delay.

In [ ]:
carrier_delay.sort_values("dep_delay", ascending=False).head()

### What changed?

The original data had one row per flight. The aggregated data has one row per carrier.

That means `dep_delay` no longer means "delay for one flight." It now means "average departure delay for that carrier."

This is why aggregation changes the meaning of a row.

### Your Turn 1

Which carrier had the lowest average arrival delay?

Use `groupby()`, `agg()`, `reset_index()`, and `sort_values()`.

In [ ]:
# Your Turn 1
# Which carrier had the lowest average arrival delay?

Write one sentence explaining what one row means in your result.

**Your answer:**  
Type your sentence here.

## 2. Multiple Aggregations

### Business question

For each carrier, what is the average departure delay, the maximum departure delay, and the number of flights?

This is useful because an average alone can hide important operational risk. A carrier may have a moderate average delay but still have some extreme delay cases.

In [ ]:
carrier_summary = flights.groupby("carrier").agg(
    avg_dep_delay=("dep_delay", "mean"),
    max_dep_delay=("dep_delay", "max"),
    flight_count=("flight", "count")
).reset_index()

carrier_summary.head()

This syntax is called named aggregation. It lets us choose clear output column names.

The pattern is:

```python
new_column_name=("original_column", "summary_function")
```

This is easier to read than a table with multi-level column names.

In [ ]:
carrier_summary.sort_values("avg_dep_delay", ascending=False).head(10)

### Your Turn 2

Create a monthly delay summary with one row per month.

Your result should include:

- average arrival delay
- median arrival delay
- number of flights

Then sort by average arrival delay from highest to lowest.

In [ ]:
# Your Turn 2
# Create one row per month with average arrival delay, median arrival delay, and flight count.

Write one sentence explaining what one row means in your monthly summary.

**Your answer:**  
Type your sentence here.

## 3. Aggregating at Two Levels: Carrier and Month

### Business question

Do some carriers experience worse delays in certain months?

Now we want one row per carrier-month combination. This is a different unit of analysis from one row per carrier or one row per month.

In [ ]:
carrier_month_delay = flights.groupby(["carrier", "month"]).agg(
    avg_arr_delay=("arr_delay", "mean"),
    flight_count=("flight", "count")
).reset_index()

carrier_month_delay.head()

Here, the grouping columns are `carrier` and `month`.

One row now means:

> one carrier in one month

This type of table is often the starting point for dashboards and business reports.

In [ ]:
carrier_month_delay.sort_values("avg_arr_delay", ascending=False).head(10)

### Light validation after aggregation

After aggregation, we should check the basic structure of the result.

For this table, each carrier-month combination should appear only once.

In [ ]:
carrier_month_delay.shape

In [ ]:
carrier_month_delay[["carrier", "month"]].duplicated().sum()

In [ ]:
carrier_month_delay.isna().sum()

These are light validation checks. We are not doing a full final data audit yet. We are simply checking whether the aggregation result has the structure we intended.

### Your Turn 3

Create a route-level summary with one row per `origin` and `dest`.

Your result should include:

- number of flights
- average arrival delay
- average distance

Then sort to find the busiest routes.

In [ ]:
# Your Turn 3
# Create one row per origin-destination route.

Run a light validation check: do any origin-destination pairs appear more than once in your route summary?

In [ ]:
# Your Turn 3 validation check

## 4. Wide vs Long Data

Aggregation summarizes data. Reshaping changes how the same values are arranged.

A wide table uses multiple columns to store different categories or time periods. A long table stores those categories or time periods in a variable column.

Both formats can be useful.

- Wide data is often easier for people to read in a report.
- Long data is often easier for plotting, grouping, and modeling workflows.

## 5. Reshaping Wide to Long with `melt()`

### Business question

Suppose a manager gives us a small report where each month is stored as a separate column. How can we reshape it so each row represents one carrier-month?

We will create a small wide example using carrier-level monthly average arrival delays.

In [ ]:
monthly_carrier = flights.groupby(["carrier", "month"]).agg(
    avg_arr_delay=("arr_delay", "mean")
).reset_index()

monthly_carrier.head()

In [ ]:
small_summary = monthly_carrier.query("carrier in ['AA', 'DL', 'UA'] and month in [1, 2, 3]")
small_summary

In [ ]:
wide_delay = small_summary.pivot(
    index="carrier",
    columns="month",
    values="avg_arr_delay"
).reset_index()

wide_delay

This wide table is easy to read: each row is a carrier, and each month is a separate column.

But if we want to plot month as a variable or group by month again, long format is usually better.

Now we use `melt()`.

In [ ]:
long_delay = wide_delay.melt(
    id_vars="carrier",
    var_name="month",
    value_name="avg_arr_delay"
)

long_delay

Important arguments in `melt()`:

- `id_vars`: the column or columns that should stay fixed.
- `var_name`: the name of the new column that stores the old column names.
- `value_name`: the name of the new column that stores the old cell values.

After melting, one row means one carrier-month.

### Your Turn 4

Use `melt()` on `wide_delay`.

Create a new long DataFrame called `long_delay_practice`.

Use:

- `carrier` as the ID variable
- `delay_month` as the variable name
- `average_delay` as the value name

In [ ]:
# Your Turn 4
# Use melt() to reshape wide_delay from wide to long.

Run a quick check: how many rows should the long dataset have, based on the number of carriers and month columns?

In [ ]:
# Your Turn 4 light validation check

## 6. Reshaping Long to Wide with `pivot()` and `pivot_table()`

### Business question

A manager wants a compact comparison table with one row per carrier and one column per month. How do we move from long back to wide?

We can use `pivot()` when each index-column combination has only one value.

In [ ]:
wide_again = long_delay.pivot(
    index="carrier",
    columns="month",
    values="avg_arr_delay"
).reset_index()

wide_again

Important arguments in `pivot()`:

- `index`: what should define each row.
- `columns`: which values should become new columns.
- `values`: which numeric values should fill the table.

If there are duplicate combinations, `pivot()` may fail. In that case, `pivot_table()` is often safer because it can aggregate duplicates.

In [ ]:
wide_with_pivot_table = small_summary.pivot_table(
    index="carrier",
    columns="month",
    values="avg_arr_delay",
    aggfunc="mean"
).reset_index()

wide_with_pivot_table

### Your Turn 5

Use the `carrier_month_delay` table from earlier.

Create a wide table with:

- one row per carrier
- one column per month
- values equal to average arrival delay

Use `pivot_table()`.

In [ ]:
# Your Turn 5
# Create a wide carrier-month delay table using pivot_table().

Run a light validation check:

- How many rows does your wide table have?
- Does that match the number of unique carriers?

In [ ]:
# Your Turn 5 light validation check

## 7. Light Validation Checklist

After aggregation or reshaping, always ask:

1. What does one row mean now?
2. Did the number of rows change in a way that makes sense?
3. Are the intended keys unique?
4. Did missing values appear or disappear?
5. Do the column names still clearly describe the values?

These checks are small, but they catch many common mistakes.

In [ ]:
long_delay[["carrier", "month"]].duplicated().sum()

In [ ]:
long_delay.isna().sum()

In [ ]:
long_delay[["carrier", "month"]].drop_duplicates().shape[0]

## 8. Final Practice: Carrier-Month Dashboard Table

Create a table named `carrier_month_summary` with one row per carrier-month and:

- `avg_departure_delay`
- `avg_arrival_delay`
- `flight_count`
- `avg_distance`

Then create `carrier_month_summary_sorted` with the highest average arrival delays first. Confirm that carrier-month keys are unique and inspect missing values.

In [ ]:
# Final Practice Step 1
# Create carrier_month_summary with one row per carrier-month.

In [ ]:
# Final Practice Step 2
# Create carrier_month_summary_sorted with highest average arrival delay first.

In [ ]:
# Final Practice Step 3
# Check for duplicate carrier-month keys. The expected result is 0.

In [ ]:
# Final Practice Step 4
# Check missing values in carrier_month_summary.

In [ ]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

if "carrier_month_summary" not in globals():
    print("Complete Final Practice Step 1 before exporting.")
else:
    module7_output_path = output_dir / "carrier_month_summary_module7.csv"
    carrier_month_summary.to_csv(module7_output_path, index=False)
    print(f"Saved {len(carrier_month_summary)} rows to: {module7_output_path}")

### Final Reflection

Write 3-5 sentences answering the following:

1. What does one row mean in `carrier_month_summary`?
2. Which transformation changed the row meaning the most?
3. What validation evidence supports the table's intended structure?
4. How could this table support an operations decision?

**Your reflection:**  
Type your answer here.

## 9. Offline Assignment - Route Performance Summary

An airport operations manager wants to review established routes rather than individual flights.

1. Create `route_performance` with one row per `origin`-`dest` pair.
2. Include flight count, average arrival delay, and average distance using named aggregation.
3. Keep routes with at least 500 flights and sort from highest to lowest average arrival delay.
4. Verify that route keys are unique and inspect missing values.
5. Save `data/processed/route_performance_module7.csv`.

In [ ]:
# Offline Assignment - Your code here
# Suggested sequence:
# 1. Start with flights.groupby(["origin", "dest"], as_index=False).agg(...).
# 2. Use named aggregation for the three requested measures.
# 3. Filter flight_count, sort avg_arrival_delay, and reset the index.
# 4. Check duplicate keys and missing values before calling .to_csv().
# Build, validate, display, and save route_performance.


### Route Summary Interpretation

**Your response here:** Write 4-5 sentences explaining what one row represents, how the 500-flight filter changes the scope, what validation evidence supports the table, and one decision the summary can support without claiming to explain the cause of delays.

## 10. Save Your Work

Before submitting:

1. Complete all Your Turn and final-practice cells.
2. Complete the offline route summary and both reflections.
3. Restart the kernel and run all cells from top to bottom.
4. Confirm both Module 7 CSV files exist in `data/processed`.
5. Save the notebook, then commit and push your work.

Suggested commit message:

```bash
git add .
git commit -m "Complete Module 7 aggregation and reshaping lab"
git push
```